# Simulator - Synthetic Organized Network Telemetry

依據「三、資料模擬設計構想」建立可重現的 network telemetry CSV，模擬真實監控資料經過整理後的欄位形式。資料概念對應 LibreNMS / RRDTool / SNMP polling，但 notebook 產出的是課程使用的 CSV。

設計重點：
- 每 300 秒一筆，模擬 30 天資料。
- 多個 device / port，各自有不同 baseline、In/Out Ratio與平均封包大小。
- 正常資料包含日週期、工作日/假日差異、隨機波動、正常業務尖峰。
- 注入 Event A-J，讓後續 Notebook 能練習 AD、Alert Reduction、Forecasting and RCA。

## Course architecture boundary

本課程有兩條資料路徑，請不要混在一起：

- **Actual operating path**: OS / network signals -> exporter -> Prometheus -> Grafana.
- **Python learning path**: organized telemetry CSV or previous notebook output -> Python analysis -> result CSV -> `outputs/prometheus-dropzone/current_results.csv` -> `python_results_exporter` -> Prometheus -> Grafana.

除非 cell 明確寫的是 operational deployment example，Python anomaly / forecasting / RCA notebooks 的主要輸入是 organized telemetry CSV 或前一個 notebook 的輸出，不是 Prometheus query。Grafana 也不直接讀 CSV；Grafana 查 Prometheus。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "environment.yml").exists():
    raise FileNotFoundError("Run this notebook from inside the course repository.")

RAW_DIR = PROJECT_ROOT / "data" / "synthetic"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "self-study"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
POLL_SECONDS = 300
START = "2026-02-01 00:00:00"
DAYS = 30
timestamps = pd.date_range(START, periods=DAYS * 24 * 12, freq=f"{POLL_SECONDS}s")

ports = pd.DataFrame(
    [
        ("edge-fw-01", "port-id7427", "wan-primary", 6.8e7, 0.72, 900, 0.95),
        ("edge-fw-01", "port-id7428", "wan-secondary", 2.6e7, 0.55, 820, 0.70),
        ("core-sw-01", "port-id7429", "server-uplink", 8.2e7, 0.38, 1050, 1.15),
        ("core-sw-01", "port-id7430", "office-vlan", 3.4e7, 0.64, 760, 0.82),
        ("dist-sw-02", "port-id7431", "backup-storage", 4.8e7, 0.44, 1180, 1.05),
    ],
    columns=["device_id", "port_id", "port_role", "base_octets", "in_share", "avg_packet_size", "business_weight"],
)
ports

## 正常Traffic模型

`traffic(t) = baseline + daily_seasonality + weekly_effect + random_noise`

這裡用白天高峰、週末折減、lognormal 噪聲與少量正常尖峰來產生合理但不過度平滑的時間序列。

In [ ]:
def daily_profile(index: pd.DatetimeIndex) -> np.ndarray:
    hour = index.hour + index.minute / 60
    morning = np.exp(-0.5 * ((hour - 10.5) / 2.6) ** 2)
    afternoon = np.exp(-0.5 * ((hour - 15.5) / 3.2) ** 2)
    evening_low = 0.25 + 0.75 / (1 + np.exp(-(hour - 7)))
    night_drop = 0.35 + 0.65 / (1 + np.exp(hour - 21))
    return (0.45 + 0.40 * morning + 0.55 * afternoon) * evening_low * night_drop


def weekly_profile(index: pd.DatetimeIndex) -> np.ndarray:
    is_weekend = index.dayofweek >= 5
    return np.where(is_weekend, 0.58, 1.0)


def normal_spikes(n: int, probability: float = 0.006) -> np.ndarray:
    spikes = np.ones(n)
    starts = np.flatnonzero(rng.random(n) < probability)
    for start in starts:
        width = int(rng.integers(1, 5))
        multiplier = float(rng.uniform(1.15, 1.75))
        spikes[start : start + width] *= multiplier
    return spikes


def build_normal_port_series(port: pd.Series) -> pd.DataFrame:
    n = len(timestamps)
    seasonal = daily_profile(timestamps) * weekly_profile(timestamps)
    noise = rng.lognormal(mean=0.0, sigma=0.10, size=n)
    traffic = port.base_octets * port.business_weight * seasonal * noise * normal_spikes(n)
    traffic = np.maximum(traffic, port.base_octets * 0.04)

    asym = np.clip(port.in_share + rng.normal(0, 0.035, n), 0.18, 0.86)
    in_octets = traffic * asym
    out_octets = traffic * (1 - asym)

    packet_size = np.clip(rng.normal(port.avg_packet_size, port.avg_packet_size * 0.08, n), 220, 1450)
    in_ucast = in_octets / packet_size
    out_ucast = out_octets / packet_size

    nucast_base = np.maximum((in_ucast + out_ucast) * rng.uniform(0.008, 0.018), 1)
    broadcast_base = np.maximum((in_ucast + out_ucast) * rng.uniform(0.0015, 0.004), 0.2)
    multicast_base = np.maximum((in_ucast + out_ucast) * rng.uniform(0.002, 0.006), 0.2)

    df = pd.DataFrame(
        {
            "timestamp": timestamps,
            "device_id": port.device_id,
            "port_id": port.port_id,
            "port_role": port.port_role,
            "INOCTETS": in_octets,
            "OUTOCTETS": out_octets,
            "INERRORS": rng.poisson(0.015, n),
            "OUTERRORS": rng.poisson(0.010, n),
            "INUCASTPKTS": in_ucast,
            "OUTUCASTPKTS": out_ucast,
            "INNUCASTPKTS": nucast_base * rng.uniform(0.45, 0.60),
            "OUTNUCASTPKTS": nucast_base * rng.uniform(0.40, 0.55),
            "INDISCARDS": rng.poisson(0.04, n),
            "OUTDISCARDS": rng.poisson(0.05, n),
            "INUNKNOWNPROTOS": rng.poisson(0.01, n),
            "INBROADCASTPKTS": broadcast_base * rng.uniform(0.50, 0.65),
            "OUTBROADCASTPKTS": broadcast_base * rng.uniform(0.35, 0.50),
            "INMULTICASTPKTS": multicast_base * rng.uniform(0.50, 0.65),
            "OUTMULTICASTPKTS": multicast_base * rng.uniform(0.35, 0.50),
        }
    )
    return df


metrics = pd.concat([build_normal_port_series(row) for _, row in ports.iterrows()], ignore_index=True)
metrics["event_label"] = "normal"
metrics["event_id"] = ""
metrics.head()

## 異常事件 A-J

每個事件都透過多欄位同步變化呈現，以便後續區分「正常業務增加」與「故障異常」。

In [ ]:
event_specs = [
    ("A", "business_traffic_growth", "port-id7430", "2026-02-04 09:00", 24, "正常業務Traffic增加"),
    ("B", "small_packet_scan", "port-id7428", "2026-02-06 13:10", 10, "大量小封包 / 掃描"),
    ("C", "large_file_backup", "port-id7431", "2026-02-08 01:00", 36, "大檔案傳輸 / 備份"),
    ("D", "queue_congestion", "port-id7427", "2026-02-11 15:20", 14, "頻寬壅塞 / Queue 滿"),
    ("E", "link_quality_issue", "port-id7429", "2026-02-14 10:35", 8, "線路品質問題"),
    ("F", "load_sensitive_link_issue", "port-id7427", "2026-02-17 16:00", 18, "高負載下線路不穩"),
    ("G", "broadcast_storm", "MULTI", "2026-02-19 11:30", 12, "Broadcast Storm / L2 Loop"),
    ("H", "multicast_flooding", "MULTI", "2026-02-22 20:05", 18, "Multicast Flooding"),
    ("I", "abnormal_device_sender", "port-id7430", "2026-02-25 09:45", 16, "異常設備大量發送"),
    ("J", "unknown_protocol_scan", "port-id7428", "2026-02-27 14:15", 10, "Unknown Protocol"),
]


def event_mask(df: pd.DataFrame, port_id: str, start: str, periods: int) -> pd.Series:
    start_ts = pd.Timestamp(start)
    end_ts = start_ts + pd.Timedelta(minutes=5 * (periods - 1))
    time_mask = df["timestamp"].between(start_ts, end_ts)
    if port_id == "MULTI":
        return time_mask
    return time_mask & (df["port_id"] == port_id)


def apply_event(df: pd.DataFrame, spec: tuple[str, str, str, str, int, str]) -> None:
    event_id, event_type, port_id, start, periods, description = spec
    mask = event_mask(df, port_id, start, periods)
    idx = df.index[mask]
    if len(idx) == 0:
        return

    ramp = np.linspace(0.8, 1.0, len(idx))
    df.loc[idx, "event_label"] = event_type
    df.loc[idx, "event_id"] = event_id

    if event_id == "A":
        df.loc[idx, ["INOCTETS", "OUTOCTETS", "INUCASTPKTS", "OUTUCASTPKTS"]] *= 1.9
    elif event_id == "B":
        df.loc[idx, ["INUCASTPKTS", "OUTUCASTPKTS"]] *= 8.0
        df.loc[idx, ["INOCTETS", "OUTOCTETS"]] *= 1.25
    elif event_id == "C":
        df.loc[idx, ["INOCTETS", "OUTOCTETS"]] *= 5.2
        df.loc[idx, ["INUCASTPKTS", "OUTUCASTPKTS"]] *= 2.0
    elif event_id == "D":
        df.loc[idx, ["INOCTETS", "OUTOCTETS", "INUCASTPKTS", "OUTUCASTPKTS"]] *= 3.8
        discard_spike = np.round(600 * ramp + rng.normal(0, 30, len(idx))).clip(80)
        df.loc[idx, ["INDISCARDS", "OUTDISCARDS"]] += np.column_stack([discard_spike, discard_spike])
    elif event_id == "E":
        error_spike = rng.poisson(180, (len(idx), 2)) + np.round(100 * ramp)[:, None]
        df.loc[idx, ["INERRORS", "OUTERRORS"]] += error_spike
    elif event_id == "F":
        df.loc[idx, ["INOCTETS", "OUTOCTETS"]] *= 2.8
        late = idx[int(len(idx) * 0.35) :]
        error_spike = (
            rng.poisson(120, (len(late), 2))
            + np.round(120 * np.linspace(0.5, 1.2, len(late)))[:, None]
        )
        df.loc[late, ["INERRORS", "OUTERRORS"]] += error_spike
    elif event_id == "G":
        df.loc[idx, ["INBROADCASTPKTS", "OUTBROADCASTPKTS", "INNUCASTPKTS", "OUTNUCASTPKTS"]] *= 18
        df.loc[idx, ["INDISCARDS", "OUTDISCARDS"]] += rng.poisson(80, (len(idx), 2))
    elif event_id == "H":
        df.loc[idx, ["INMULTICASTPKTS", "OUTMULTICASTPKTS"]] *= 16
        df.loc[idx, ["INDISCARDS", "OUTDISCARDS"]] += rng.poisson(45, (len(idx), 2))
    elif event_id == "I":
        df.loc[idx, ["OUTOCTETS", "OUTUCASTPKTS", "OUTNUCASTPKTS"]] *= 5.5
        df.loc[idx, ["OUTBROADCASTPKTS", "OUTMULTICASTPKTS"]] *= 2.5
    elif event_id == "J":
        df.loc[idx, "INUNKNOWNPROTOS"] += rng.poisson(360, len(idx)) + 80
        df.loc[idx, "INUCASTPKTS"] *= 2.4


for spec in event_specs:
    apply_event(metrics, spec)

event_catalog = pd.DataFrame(
    event_specs,
    columns=["event_id", "event_type", "port_id", "start_time", "periods_5min", "description"],
)
event_catalog["end_time"] = pd.to_datetime(event_catalog["start_time"]) + pd.to_timedelta((event_catalog["periods_5min"] - 1) * 5, unit="m")
event_catalog

In [ ]:
metric_columns = [
    "INOCTETS", "OUTOCTETS",
    "INERRORS", "OUTERRORS",
    "INUCASTPKTS", "OUTUCASTPKTS",
    "INNUCASTPKTS", "OUTNUCASTPKTS",
    "INDISCARDS", "OUTDISCARDS",
    "INUNKNOWNPROTOS",
    "INBROADCASTPKTS", "OUTBROADCASTPKTS",
    "INMULTICASTPKTS", "OUTMULTICASTPKTS",
]

metrics[metric_columns] = metrics[metric_columns].clip(lower=0)
counter_like = [c for c in metric_columns if c not in ["INOCTETS", "OUTOCTETS"]]
metrics[counter_like] = metrics[counter_like].round().astype(int)
metrics[["INOCTETS", "OUTOCTETS"]] = metrics[["INOCTETS", "OUTOCTETS"]].round(2)

metrics = metrics.sort_values(["device_id", "port_id", "timestamp"]).reset_index(drop=True)
metrics.to_csv(RAW_DIR / "synthetic_rrd_metrics.csv", index=False)
event_catalog.to_csv(RAW_DIR / "synthetic_event_catalog.csv", index=False)

print(f"rows: {len(metrics):,}")
print(f"ports: {metrics['port_id'].nunique()}")
print(f"time range: {metrics['timestamp'].min()} -> {metrics['timestamp'].max()}")
print(f"saved: {RAW_DIR / 'synthetic_rrd_metrics.csv'}")
metrics.groupby("event_label").size().sort_values(ascending=False)

In [ ]:
sample_port = "port-id7427"
plot_df = metrics[metrics["port_id"] == sample_port].copy()
plot_df["traffic_total"] = plot_df["INOCTETS"] + plot_df["OUTOCTETS"]
plot_df["errors_total"] = plot_df["INERRORS"] + plot_df["OUTERRORS"]
plot_df["discards_total"] = plot_df["INDISCARDS"] + plot_df["OUTDISCARDS"]

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(plot_df["timestamp"], plot_df["traffic_total"], label="traffic_total")
    axes[1].plot(plot_df["timestamp"], plot_df["errors_total"], color="tab:red", label="errors_total")
    axes[2].plot(plot_df["timestamp"], plot_df["discards_total"], color="tab:orange", label="discards_total")
    for ax in axes:
        ax.legend(loc="upper left")
        ax.grid(alpha=0.25)
    fig.suptitle(f"Synthetic organized network telemetry - {sample_port}")
    plt.tight_layout()
except ImportError:
    display_cols = ["timestamp", "traffic_total", "errors_total", "discards_total", "event_label"]
    print("matplotlib not installed; showing event-window sample instead.")
    plot_df.loc[plot_df["event_label"] != "normal", display_cols].head(12)